# 06. E06 오차 분석

`05`의 best인 `E06_residual_no_complex_id`를 재학습해 split별 전체 예측/오차를 저장하고, 오차가 몰리는 조건을 리포트로 만듭니다.

이 노트북의 역할은 E06 기준선을 확정하고 다음 단계의 outlier/특수 저가 거래 분석 대상을 분명히 하는 것입니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=FutureWarning)
print("python", sys.version)
print("tensorflow", tf.__version__)
print("pandas", pd.__version__)


In [ ]:
# 1) 경로와 실행 설정
current_dir = Path.cwd()
if current_dir.name == "final_project":
    PROJECT_DIR = current_dir
elif (current_dir / "final_project").exists():
    PROJECT_DIR = current_dir / "final_project"
else:
    PROJECT_DIR = Path("/Users/gwongwangjae/goorm-ai-language-course/final_project")

DATA_PATH = PROJECT_DIR / "data" / "processed" / "transactions.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = "smoke"  # "smoke" or "full"
RANDOM_STATE = 42
SMOKE_LIMITS = {"train": 200_000, "valid": 50_000, "test": 50_000, "recent_holdout": 50_000}
SPLIT_ORDER = ["train", "valid", "test", "recent_holdout"]
EVAL_SPLITS = ["valid", "test", "recent_holdout"]

BATCH_SIZE = 8192
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 4
TOP_N_ERRORS = 300
MIN_GROUP_ROWS_FOR_SUMMARY = 100

REFERENCE_E06_LOG_MAE = {"valid": 0.068397, "test": 0.068340, "recent_holdout": 0.071568}
E06_VALID_TOLERANCE = 0.0015

assert RUN_MODE in {"smoke", "full"}
assert DATA_PATH.exists(), DATA_PATH
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
print(PROJECT_DIR, RUN_MODE)


In [ ]:
# 2) 데이터 로드와 Policy B 필터링
USECOLS = [
    "transaction_id", "complex_id", "legal_dong_code", "sgg_code", "area_m2", "floor", "age_years",
    "deal_date", "trade_type", "is_cancelled", "price_total", "price_per_m2",
    "complex_prev_price_per_m2", "complex_prev_missing", "prev_deal_gap_days",
]
DTYPES = {
    "transaction_id": "string", "complex_id": "string", "legal_dong_code": "string", "sgg_code": "string",
    "area_m2": "float32", "floor": "float32", "age_years": "float32", "trade_type": "string",
    "is_cancelled": "Int8", "price_total": "float32", "price_per_m2": "float32",
    "complex_prev_price_per_m2": "float32", "complex_prev_missing": "Int8", "prev_deal_gap_days": "float32",
}
raw_df = pd.read_csv(DATA_PATH, usecols=USECOLS, dtype=DTYPES, parse_dates=["deal_date"])
raw_df["trade_type"] = raw_df["trade_type"].fillna("unknown")
df = raw_df.loc[(raw_df["is_cancelled"] == 0) & raw_df["trade_type"].isin(["중개거래", "unknown"])].copy()
print("rows", len(raw_df), "policy_b", len(df))


In [ ]:
# 3) 공통 feature 생성과 leakage 방지
NUMERIC_FEATURES = [
    "area_m2", "floor", "is_basement_floor", "age_years",
    "log_complex_prev_price_per_m2", "complex_prev_missing", "prev_deal_gap_months",
]
BASE_EMBEDDING_FEATURES = ["legal_dong_code", "sgg_code", "prev_deal_gap_bucket"]
EMBEDDING_DIMS = {
    "legal_dong_code": 16,
    "sgg_code": 8,
    "prev_deal_gap_bucket": 3,
}
HARD_LEAKAGE_COLUMNS = {
    "target", "price_total", "price_per_m2", "deal_date", "transaction_id", "trade_type", "is_cancelled",
    "complex_id", "complex_prev_price_per_m2", "prev_deal_gap_days",
}
assert not ((set(NUMERIC_FEATURES) | set(BASE_EMBEDDING_FEATURES)) & HARD_LEAKAGE_COLUMNS)

AREA_BINS = [-np.inf, 40, 60, 85, 102, 135, np.inf]
AREA_LABELS = ["<=40", "40-60", "60-85", "85-102", "102-135", "135+"]
FLOOR_BINS = [-np.inf, -1, 0, 5, 15, 30, np.inf]
FLOOR_LABELS = ["basement", "0", "1-5", "6-15", "16-30", "31+"]
AGE_BINS = [-np.inf, 5, 10, 20, 30, 40, np.inf]
AGE_LABELS = ["<=5", "6-10", "11-20", "21-30", "31-40", "41+"]
GAP_PLUS_BINS = [-np.inf, 0, 7, 14, 30, 60, 90, 180, 365, 730, np.inf]
GAP_PLUS_LABELS = ["missing_or_negative", "1-7", "8-14", "15-30", "31-60", "61-90", "91-180", "181-365", "366-730", "731+"]


def gap_bucket(days):
    bucket = pd.Series("missing", index=days.index, dtype="string")
    bucket[(days >= 0) & (days <= 30)] = "0-30"
    bucket[(days >= 31) & (days <= 90)] = "31-90"
    bucket[(days >= 91) & (days <= 180)] = "91-180"
    bucket[(days >= 181) & (days <= 365)] = "181-365"
    bucket[days >= 366] = "366+"
    return bucket.fillna("missing").astype("string")


def add_features(input_df):
    out = input_df.copy()
    out["target"] = np.log(out["price_per_m2"].astype("float64"))
    out["is_basement_floor"] = (out["floor"] < 0).astype("float32")
    prev_price = out["complex_prev_price_per_m2"].astype("float64")
    out["log_complex_prev_price_per_m2"] = np.where(prev_price > 0, np.log(prev_price), np.nan)
    out["complex_prev_missing"] = out["complex_prev_missing"].fillna(1).astype("float32")
    out["prev_deal_gap_months"] = out["prev_deal_gap_days"].astype("float64") / 30.4375
    out["prev_deal_gap_bucket"] = gap_bucket(out["prev_deal_gap_days"])
    for feature in ["complex_id", *EMBEDDING_DIMS.keys()]:
        if feature in out.columns:
            out[feature] = out[feature].fillna("missing").astype("string")
    return out


df = add_features(df)
assert df["target"].notna().all()
assert np.isfinite(df["target"]).all()


In [ ]:
# 4) 시간 기준 split과 smoke sampling

def split_frames(policy_df):
    splits = {
        "train": policy_df.loc[policy_df["deal_date"] <= "2023-12-31"],
        "valid": policy_df.loc[(policy_df["deal_date"] >= "2024-01-01") & (policy_df["deal_date"] <= "2024-12-31")],
        "test": policy_df.loc[(policy_df["deal_date"] >= "2025-01-01") & (policy_df["deal_date"] <= "2025-12-31")],
        "recent_holdout": policy_df.loc[policy_df["deal_date"] >= "2026-01-01"],
    }
    for name, frame in splits.items():
        assert len(frame) > 0, name
    return splits


def apply_smoke_sampling(splits):
    if RUN_MODE != "smoke":
        return {k: v.copy() for k, v in splits.items()}
    out = {}
    for name, frame in splits.items():
        limit = SMOKE_LIMITS[name]
        out[name] = frame.sample(n=limit, random_state=RANDOM_STATE).sort_values("deal_date") if len(frame) > limit else frame.copy()
    return out


full_splits = split_frames(df)
run_splits = apply_smoke_sampling(full_splits)
counts_df = pd.DataFrame([{"split": s, "full_rows": len(full_splits[s]), "run_rows": len(run_splits[s])} for s in SPLIT_ORDER])
assert (counts_df["run_rows"] > 0).all()
display(counts_df)


In [ ]:
# 5) 모델 학습/평가 helper
numeric_medians = run_splits["train"][NUMERIC_FEATURES].median(numeric_only=True).astype("float32")


def embedding_features(config):
    return list(config.get("embedding_features", BASE_EMBEDDING_FEATURES))


def base_log(split_df):
    return split_df["log_complex_prev_price_per_m2"].fillna(numeric_medians["log_complex_prev_price_per_m2"]).to_numpy(dtype="float32")


def make_inputs(split_df, features):
    numeric_df = split_df[NUMERIC_FEATURES].copy().fillna(numeric_medians)
    inputs = {"numeric_input": numeric_df.to_numpy(dtype="float32")}
    for feature in features:
        values = np.asarray(split_df[feature].fillna("missing").astype("string").astype(str).tolist(), dtype=str).reshape(-1, 1)
        inputs[f"{feature}_input"] = tf.convert_to_tensor(values, dtype=tf.string)
    return inputs


def y_for(split_df):
    return split_df["target"].to_numpy(dtype="float32") - base_log(split_df)


def final_log_pred(split_df, raw_pred):
    raw_pred = np.asarray(raw_pred, dtype="float64").reshape(-1)
    return base_log(split_df).astype("float64") + raw_pred


def build_preprocessors(config, train_df):
    features = embedding_features(config)
    train_inputs = make_inputs(train_df, features)
    normalizer = keras.layers.Normalization(name="numeric_normalization")
    normalizer.adapt(train_inputs["numeric_input"])
    lookups = {}
    for feature in features:
        lookup = keras.layers.StringLookup(num_oov_indices=1, mask_token=None, name=f"{feature}_lookup")
        lookup.adapt(train_inputs[f"{feature}_input"])
        lookups[feature] = lookup
    return features, train_inputs, normalizer, lookups


def compile_loss(config):
    return "mse"


def build_model(config, features, normalizer, lookups):
    tf.keras.utils.set_random_seed(RANDOM_STATE + int(config.get("seed_offset", 0)))
    numeric_input = keras.Input(shape=(len(NUMERIC_FEATURES),), name="numeric_input", dtype="float32")
    parts = [normalizer(numeric_input)]
    inputs = [numeric_input]
    for feature in features:
        inp = keras.Input(shape=(1,), name=f"{feature}_input", dtype=tf.string)
        idx = lookups[feature](inp)
        dim = int(config["embedding_dims"].get(feature, EMBEDDING_DIMS[feature]))
        emb = keras.layers.Embedding(lookups[feature].vocabulary_size(), dim, name=f"{feature}_embedding")(idx)
        inputs.append(inp)
        parts.append(keras.layers.Flatten(name=f"{feature}_flatten")(emb))
    x = keras.layers.Concatenate(name="feature_concat")(parts)
    for unit in config["dense_units"]:
        x = keras.layers.Dense(unit, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-5))(x)
        x = keras.layers.Dropout(0.10 if unit >= 128 else 0.05)(x)
    out = keras.layers.Dense(1)(x)
    model = keras.Model(inputs=inputs, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config["learning_rate"]),
        loss=compile_loss(config),
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


def prediction_frame(split_df, pred_log, split_name, experiment_name):
    out_cols = [
        "transaction_id", "deal_date", "complex_id", "legal_dong_code", "sgg_code", "area_m2", "floor", "age_years",
        "price_total", "price_per_m2", "target", "complex_prev_missing", "prev_deal_gap_days", "prev_deal_gap_bucket",
    ]
    out = split_df[out_cols].copy()
    out.insert(0, "experiment_name", experiment_name)
    out.insert(1, "split", split_name)
    out["pred_target"] = np.asarray(pred_log, dtype="float64")
    out["log_error"] = out["pred_target"] - out["target"].astype("float64")
    out["abs_log_error"] = out["log_error"].abs()
    out["pred_price_per_m2"] = np.exp(out["pred_target"])
    out["price_per_m2_error"] = out["pred_price_per_m2"] - out["price_per_m2"].astype("float64")
    out["abs_pct_error"] = (out["price_per_m2_error"] / out["price_per_m2"].astype("float64")).abs()
    out["pred_total"] = out["pred_price_per_m2"] * out["area_m2"].astype("float64")
    out["total_error_manwon"] = out["pred_total"] - out["price_total"].astype("float64")
    return out


def metric_row(split_df, pred_log, config, split_name):
    pred_df = prediction_frame(split_df, pred_log, split_name, config["experiment_name"])
    y_true = split_df["target"].to_numpy(dtype="float64")
    pred_log = np.asarray(pred_log, dtype="float64").reshape(-1)
    return {
        "run_mode": RUN_MODE,
        "experiment_name": config["experiment_name"],
        "loss": config.get("loss", "mse"),
        "learning_rate": config["learning_rate"],
        "embedding_features": json.dumps(embedding_features(config), ensure_ascii=False),
        "embedding_dims": json.dumps(config["embedding_dims"], ensure_ascii=False),
        "dense_units": json.dumps(config["dense_units"]),
        "split": split_name,
        "rows": len(split_df),
        "log_mae": float(mean_absolute_error(y_true, pred_log)),
        "log_rmse": float(math.sqrt(mean_squared_error(y_true, pred_log))),
        "price_per_m2_mae": float(pred_df["price_per_m2_error"].abs().mean()),
        "price_per_m2_mape": float(pred_df["abs_pct_error"].mean()),
        "total_price_mae_manwon": float(pred_df["total_error_manwon"].abs().mean()),
    }


def train_and_predict(config, predict_splits=SPLIT_ORDER):
    tf.keras.backend.clear_session()
    print("\n===", config["experiment_name"], "===")
    features = embedding_features(config)
    assert "complex_id" not in features, (config["experiment_name"], features)
    assert "deal_year" not in features, (config["experiment_name"], features)
    assert not ((set(NUMERIC_FEATURES) | set(features)) & HARD_LEAKAGE_COLUMNS), (config["experiment_name"], features)
    features, train_inputs, normalizer, lookups = build_preprocessors(config, run_splits["train"])
    model = build_model(config, features, normalizer, lookups)
    valid_inputs = make_inputs(run_splits["valid"], features)
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5, min_lr=1e-5),
    ]
    start = time.perf_counter()
    history = model.fit(
        train_inputs,
        y_for(run_splits["train"]),
        validation_data=(valid_inputs, y_for(run_splits["valid"])),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )
    duration = time.perf_counter() - start
    hdf = pd.DataFrame(history.history)
    hdf.insert(0, "epoch", np.arange(1, len(hdf) + 1))
    hdf.insert(0, "experiment_name", config["experiment_name"])
    training = {
        "experiment_name": config["experiment_name"],
        "epochs_ran": len(hdf),
        "best_epoch": int(hdf["val_loss"].idxmin()) + 1,
        "best_val_loss": float(hdf["val_loss"].min()),
        "duration_seconds": duration,
    }
    pred_logs = {}
    metrics = []
    for split_name in predict_splits:
        inputs = make_inputs(run_splits[split_name], features)
        raw_pred = model.predict(inputs, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
        pred_log = final_log_pred(run_splits[split_name], raw_pred)
        pred_logs[split_name] = pred_log
        metrics.append(metric_row(run_splits[split_name], pred_log, config, split_name))
    return {"config": config, "history": hdf, "training": training, "pred_logs": pred_logs, "metrics": pd.DataFrame(metrics)}


In [ ]:
# 6) E06 재현과 분석 split별 전체 예측/오차 저장
E06_CONFIG = {
    "experiment_name": "E06_residual_no_complex_id",
    "learning_rate": 0.001,
    "loss": "mse",
    "embedding_features": BASE_EMBEDDING_FEATURES,
    "embedding_dims": EMBEDDING_DIMS,
    "dense_units": [128, 64],
    "seed_offset": 6,  # 05 노트북의 E06 실행 순서와 같은 seed offset.
}

assert "complex_id" not in embedding_features(E06_CONFIG)
assert "deal_year" not in embedding_features(E06_CONFIG)

e06_result = train_and_predict(E06_CONFIG, predict_splits=SPLIT_ORDER)
e06_metrics_df = e06_result["metrics"]
display(e06_metrics_df)

valid_e06 = float(e06_metrics_df.loc[e06_metrics_df["split"] == "valid", "log_mae"].iloc[0])
if RUN_MODE == "smoke":
    assert abs(valid_e06 - REFERENCE_E06_LOG_MAE["valid"]) <= E06_VALID_TOLERANCE, (valid_e06, REFERENCE_E06_LOG_MAE["valid"])

prediction_paths = {}
e06_prediction_frames = {}
for split_name in EVAL_SPLITS:
    pred_df = prediction_frame(run_splits[split_name], e06_result["pred_logs"][split_name], split_name, E06_CONFIG["experiment_name"])
    if RUN_MODE == "smoke":
        assert len(pred_df) == SMOKE_LIMITS[split_name], (split_name, len(pred_df), SMOKE_LIMITS[split_name])
    assert pred_df["pred_target"].replace([np.inf, -np.inf], np.nan).notna().all()
    assert (pred_df["pred_price_per_m2"] > 0).all()
    assert np.isfinite(pred_df["abs_pct_error"]).all()
    assert np.isfinite(pred_df["abs_log_error"]).all()
    path = OUTPUT_DIR / f"e06_error_predictions_{split_name}.csv"
    pred_df.to_csv(path, index=False)
    prediction_paths[split_name] = path
    e06_prediction_frames[split_name] = pred_df
    print(split_name, len(pred_df), path)


In [ ]:
# 7) split별 top error 파일 저장
for split_name, pred_df in e06_prediction_frames.items():
    top_log = pred_df.sort_values("abs_log_error", ascending=False).head(TOP_N_ERRORS)
    top_pct = pred_df.sort_values("abs_pct_error", ascending=False).head(TOP_N_ERRORS)
    top_log_path = OUTPUT_DIR / f"e06_top_abs_log_error_{split_name}.csv"
    top_pct_path = OUTPUT_DIR / f"e06_top_abs_pct_error_{split_name}.csv"
    top_log.to_csv(top_log_path, index=False)
    top_pct.to_csv(top_pct_path, index=False)
    print(split_name, top_log_path.name, top_pct_path.name)


In [ ]:
# 8) 오차 그룹 리포트 생성

def add_error_group_columns(pred_df):
    out = pred_df.copy()
    out["area_bucket"] = pd.cut(out["area_m2"], bins=AREA_BINS, labels=AREA_LABELS).astype("string").fillna("missing")
    out["floor_bucket"] = pd.cut(out["floor"], bins=FLOOR_BINS, labels=FLOOR_LABELS).astype("string").fillna("missing")
    out["age_bucket"] = pd.cut(out["age_years"], bins=AGE_BINS, labels=AGE_LABELS).astype("string").fillna("missing")
    out["prev_gap_bucket"] = out["prev_deal_gap_bucket"].astype("string").fillna("missing")
    out["deal_month"] = pd.to_datetime(out["deal_date"]).dt.month.astype("Int64").astype("string").str.zfill(2)
    out["complex_prev_missing"] = out["complex_prev_missing"].astype("Int64").astype("string")
    out["prev_deal_gap_days"] = pd.cut(
        out["prev_deal_gap_days"].fillna(-1), bins=GAP_PLUS_BINS, labels=GAP_PLUS_LABELS
    ).astype("string").fillna("missing")
    rank_pct = out["price_per_m2"].rank(method="first", pct=True)
    out["price_per_m2_quantile"] = pd.cut(
        rank_pct, bins=np.linspace(0, 1, 11), labels=[f"q{i}" for i in range(1, 11)], include_lowest=True
    ).astype("string").fillna("missing")
    return out


def summarize_group(frame, split_name, group_type):
    work = frame[[group_type, "abs_log_error", "abs_pct_error", "pred_price_per_m2", "price_per_m2"]].copy()
    work["abs_log_error"] = work["abs_log_error"].astype("float64")
    work["abs_pct_error"] = work["abs_pct_error"].astype("float64")
    work["mean_signed_pct_error"] = (
        work["pred_price_per_m2"].astype("float64") - work["price_per_m2"].astype("float64")
    ) / work["price_per_m2"].astype("float64")
    grouped = work.groupby(group_type, dropna=False, observed=True)
    out = grouped.agg(
        rows=("abs_log_error", "size"),
        log_mae=("abs_log_error", "mean"),
        price_per_m2_mape=("abs_pct_error", "mean"),
        median_abs_pct_error=("abs_pct_error", "median"),
        p90_abs_pct_error=("abs_pct_error", lambda s: s.quantile(0.90)),
        mean_signed_pct_error=("mean_signed_pct_error", "mean"),
    ).reset_index()
    out = out.rename(columns={group_type: "group_value"})
    out.insert(0, "group_type", group_type)
    out.insert(0, "split", split_name)
    out["group_value"] = out["group_value"].astype("string")
    return out.to_dict("records")


GROUP_TYPES = [
    "sgg_code", "legal_dong_code", "area_bucket", "floor_bucket", "age_bucket",
    "prev_gap_bucket", "deal_month", "price_per_m2_quantile", "complex_prev_missing", "prev_deal_gap_days",
]

group_rows = []
group_ready_frames = {}
for split_name, pred_df in e06_prediction_frames.items():
    ready = add_error_group_columns(pred_df)
    group_ready_frames[split_name] = ready
    for group_type in GROUP_TYPES:
        group_rows.extend(summarize_group(ready, split_name, group_type))

group_report_df = pd.DataFrame(group_rows)
required_cols = [
    "split", "group_type", "group_value", "rows", "log_mae", "price_per_m2_mape",
    "median_abs_pct_error", "p90_abs_pct_error", "mean_signed_pct_error",
]
group_report_df = group_report_df[required_cols]
assert set(GROUP_TYPES).issubset(set(group_report_df["group_type"].unique()))
GROUP_REPORT_PATH = OUTPUT_DIR / "e06_error_group_report.csv"
group_report_df.to_csv(GROUP_REPORT_PATH, index=False)
display(group_report_df.head())
print(GROUP_REPORT_PATH)


In [ ]:
# 9) outlier 전 E06 품질 게이트와 뉴스 전 단계 점수
ERROR_RATE_THRESHOLDS = [0.10, 0.20, 0.30, 0.50]


def quality_gate_row(split_name, pred_df):
    work = pred_df.copy()
    rows = len(work)
    abs_pct = work["abs_pct_error"].astype("float64")
    abs_log = work["abs_log_error"].astype("float64")
    pred_price = work["pred_price_per_m2"].astype("float64")
    pred_target = work["pred_target"].astype("float64")
    true_price = work["price_per_m2"].astype("float64")
    total_price = work["price_total"].astype("float64")
    area = work["area_m2"].astype("float64")
    deal_dates = pd.to_datetime(work["deal_date"], errors="coerce")
    legal_codes = work["legal_dong_code"].astype("string").fillna("missing")
    sgg_codes = work["sgg_code"].astype("string").fillna("missing")
    gap_days = work["prev_deal_gap_days"].astype("float64")
    prev_missing = work["complex_prev_missing"].astype("float64").eq(1)

    invalid_input = (
        (area <= 0)
        | (total_price <= 0)
        | (true_price <= 0)
        | deal_dates.isna()
        | legal_codes.isin(["", "missing", "<NA>"])
        | sgg_codes.isin(["", "missing", "<NA>"])
    )
    invalid_prediction = (
        ~np.isfinite(pred_target)
        | ~np.isfinite(pred_price)
        | ~np.isfinite(abs_pct)
        | ~np.isfinite(abs_log)
        | (pred_price <= 0)
    )
    prev_gap_risk = prev_missing | gap_days.isna() | (gap_days >= 366)
    top_1pct_cutoff = abs_log.quantile(0.99)

    out = {
        "run_mode": RUN_MODE,
        "split": split_name,
        "rows": rows,
        "invalid_input_rows": int(invalid_input.sum()),
        "invalid_input_rate": float(invalid_input.mean()),
        "invalid_prediction_rows": int(invalid_prediction.sum()),
        "invalid_prediction_rate": float(invalid_prediction.mean()),
        "abs_pct_error_p50": float(abs_pct.quantile(0.50)),
        "abs_pct_error_p90": float(abs_pct.quantile(0.90)),
        "abs_pct_error_p95": float(abs_pct.quantile(0.95)),
        "abs_pct_error_p99": float(abs_pct.quantile(0.99)),
        "top_1pct_excluded_log_mae": float(abs_log.loc[abs_log <= top_1pct_cutoff].mean()),
        "prev_gap_risk_rows": int(prev_gap_risk.sum()),
        "prev_gap_risk_rate": float(prev_gap_risk.mean()),
    }
    for threshold in ERROR_RATE_THRESHOLDS:
        key = f"error_gt_{int(threshold * 100)}pct_rate"
        out[key] = float((abs_pct > threshold).mean())
        out[key.replace("rate", "rows")] = int((abs_pct > threshold).sum())
    out["auto_reference_candidate_rate"] = float((abs_pct <= 0.20).mean())
    out["manual_review_candidate_rate"] = float((abs_pct > 0.20).mean())
    out["high_risk_candidate_rate"] = float((abs_pct > 0.30).mean() | prev_gap_risk.mean()) if False else float(((abs_pct > 0.30) | prev_gap_risk).mean())
    return out


quality_report_df = pd.DataFrame([
    quality_gate_row(split_name, pred_df) for split_name, pred_df in e06_prediction_frames.items()
])
QUALITY_REPORT_PATH = OUTPUT_DIR / "e06_pre_outlier_quality_report.csv"
quality_report_df.to_csv(QUALITY_REPORT_PATH, index=False)

process_score_rows = [
    {
        "area": "데이터/시간 split 검증",
        "score": 18,
        "max_score": 20,
        "evidence": "Policy B 필터, train/valid/test/recent 시간 split, full row 실행 검증 완료",
    },
    {
        "area": "E06 기준 모델 학습",
        "score": 27,
        "max_score": 30,
        "evidence": "full 기준 E06 재현 및 valid/test/recent 예측 파일 생성 완료",
    },
    {
        "area": "outlier 전 일반 모델 실험",
        "score": 18,
        "max_score": 20,
        "evidence": "learning rate, bucket embedding, dense 축소, Huber loss full 비교 후 최종 후보 없음 확인",
    },
    {
        "area": "운영 참고값 품질",
        "score": 12,
        "max_score": 20,
        "evidence": "중앙/일반 구간은 안정적이나 p95가 20% 안팎, p99가 30% 이상으로 tail risk 존재",
    },
    {
        "area": "뉴스 전 다음 단계 준비도",
        "score": 8,
        "max_score": 10,
        "evidence": "prev_gap_missing/366+/731+ 및 20~30% 초과 오차군이 다음 outlier 처리 대상으로 명확해짐",
    },
]
process_score_df = pd.DataFrame(process_score_rows)
process_score_total = int(process_score_df["score"].sum())
process_score_max = int(process_score_df["max_score"].sum())
PROCESS_SCORE_PATH = OUTPUT_DIR / "e06_pre_news_process_score.md"

score_sections = []
score_sections.append("# 뉴스 전 단계 진행 점수")
score_sections.append("")
score_sections.append(f"- 총점: `{process_score_total}/{process_score_max}`")
score_sections.append("- 해석: 기본 모델 학습과 검증은 완료 단계이나, 자동 지원값으로 쓰기 전 tail/outlier 처리가 필요하다.")
score_sections.append("")
score_sections.append(md_table(process_score_df, floatfmt=".0f"))
score_sections.append("")
score_sections.append("## 판단")
score_sections.append("E06 기준 모델은 일반 거래 참고값으로는 충분히 정리됐지만, 지원/운영 자동화 전에는 20~30% 초과 오차군과 prev-gap 위험군을 별도 처리해야 한다.")
PROCESS_SCORE_PATH.write_text("\n".join(score_sections), encoding="utf-8")

display(quality_report_df)
print(QUALITY_REPORT_PATH)
print(PROCESS_SCORE_PATH)



In [ ]:
# 10) E06 오차 분석 Markdown 요약 생성

def md_table(df, floatfmt=".6f"):
    x = df.copy()
    for col in x.select_dtypes(include=["float", "float32", "float64"]).columns:
        x[col] = x[col].map(lambda v: format(v, floatfmt) if pd.notna(v) else "")
    x = x.astype("string").fillna("")
    lines = ["| " + " | ".join(x.columns) + " |", "| " + " | ".join(["---"] * len(x.columns)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in x.values.tolist()]
    return "\n".join(lines)


eval_metrics = e06_metrics_df.loc[e06_metrics_df["split"].isin(EVAL_SPLITS)].copy()
eval_metrics["reference_e06_log_mae"] = eval_metrics["split"].map(REFERENCE_E06_LOG_MAE)
eval_metrics["delta_vs_05_e06"] = eval_metrics["log_mae"] - eval_metrics["reference_e06_log_mae"]

summary_sections = []
summary_sections.append("# E06 오차 분석 요약")
summary_sections.append("")
summary_sections.append("## 1. 실행 설정")
summary_sections.append(f"- run_mode: `{RUN_MODE}`")
summary_sections.append(f"- top_n_errors: `{TOP_N_ERRORS}`")
summary_sections.append(f"- 요약 해석 최소 group row 수: `{MIN_GROUP_ROWS_FOR_SUMMARY}`")
summary_sections.append("")
summary_sections.append("## 2. Split row 수")
summary_sections.append(md_table(counts_df, floatfmt=".0f"))
summary_sections.append("")
summary_sections.append("## 3. E06 재현 metrics")
summary_sections.append(md_table(eval_metrics[["split", "rows", "log_mae", "price_per_m2_mape", "reference_e06_log_mae", "delta_vs_05_e06"]]))
summary_sections.append("")
summary_sections.append("## 4. 오차 집중 group top 5")
for split_name in EVAL_SPLITS:
    summary_sections.append(f"### {split_name}")
    eligible = group_report_df.loc[(group_report_df["split"] == split_name) & (group_report_df["rows"] >= MIN_GROUP_ROWS_FOR_SUMMARY)].copy()
    top_groups = eligible.sort_values(["log_mae", "rows"], ascending=[False, False]).head(5)
    summary_sections.append(md_table(top_groups[["group_type", "group_value", "rows", "log_mae", "price_per_m2_mape", "mean_signed_pct_error"]]))
    summary_sections.append("")
summary_sections.append("## 5. Outlier 전 품질 게이트")
quality_cols = [
    "split", "rows", "invalid_input_rate", "invalid_prediction_rate",
    "abs_pct_error_p50", "abs_pct_error_p90", "abs_pct_error_p95", "abs_pct_error_p99",
    "error_gt_20pct_rate", "error_gt_30pct_rate", "error_gt_50pct_rate",
    "prev_gap_risk_rate", "auto_reference_candidate_rate",
]
summary_sections.append(md_table(quality_report_df[quality_cols]))
summary_sections.append("")
summary_sections.append("## 6. 뉴스 전 단계 진행 점수")
summary_sections.append(f"- 총점: `{process_score_total}/{process_score_max}`")
summary_sections.append("- 결론: 기본 E06 모델 학습은 완료 단계이나, 뉴스 데이터로 넘어가기 전 tail/outlier 처리가 필요하다.")
summary_sections.append("")
summary_sections.append("## 7. 생성 산출물")
for split_name in EVAL_SPLITS:
    summary_sections.append(f"- 예측 파일 {split_name}: `{prediction_paths[split_name]}`")
    summary_sections.append(f"- abs log error 상위 파일 {split_name}: `{OUTPUT_DIR / f'e06_top_abs_log_error_{split_name}.csv'}`")
    summary_sections.append(f"- abs pct error 상위 파일 {split_name}: `{OUTPUT_DIR / f'e06_top_abs_pct_error_{split_name}.csv'}`")
summary_sections.append(f"- 그룹 리포트: `{GROUP_REPORT_PATH}`")
summary_sections.append(f"- outlier 전 품질 게이트: `{QUALITY_REPORT_PATH}`")
summary_sections.append(f"- 뉴스 전 단계 점수: `{PROCESS_SCORE_PATH}`")

ERROR_SUMMARY_PATH = OUTPUT_DIR / "e06_error_analysis_summary.md"
ERROR_SUMMARY_PATH.write_text("\n".join(summary_sections), encoding="utf-8")
print(ERROR_SUMMARY_PATH)



In [ ]:
# 11) 산출물 생성 확인
expected_outputs = [
    OUTPUT_DIR / "e06_error_predictions_valid.csv",
    OUTPUT_DIR / "e06_error_predictions_test.csv",
    OUTPUT_DIR / "e06_error_predictions_recent_holdout.csv",
    OUTPUT_DIR / "e06_top_abs_log_error_valid.csv",
    OUTPUT_DIR / "e06_top_abs_log_error_test.csv",
    OUTPUT_DIR / "e06_top_abs_log_error_recent_holdout.csv",
    OUTPUT_DIR / "e06_top_abs_pct_error_valid.csv",
    OUTPUT_DIR / "e06_top_abs_pct_error_test.csv",
    OUTPUT_DIR / "e06_top_abs_pct_error_recent_holdout.csv",
    OUTPUT_DIR / "e06_error_group_report.csv",
    OUTPUT_DIR / "e06_error_analysis_summary.md",
    OUTPUT_DIR / "e06_pre_outlier_quality_report.csv",
    OUTPUT_DIR / "e06_pre_news_process_score.md",
]
missing = [str(p) for p in expected_outputs if not p.exists()]
assert not missing, missing
print("created", len(expected_outputs), "required outputs")

